# Data Prep: 6 - World GDP / Population / CO2 Emissions

Steps:
1. Initial Data Exploration
2. Handling Missing Values
3. Dealing with Duplicates
4. Handling Outliers
5. Data Manipulation (Lag Features)
6. Partitioning Dataset (Time-aware split)
7. Feature Scaling + Preprocessing
8. Visualizing Data


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

RAW_PATH = Path("data/raw/6-World_GDP_Population_CO2_Emissions_Dataset.csv")
assert RAW_PATH.exists(), f"Missing dataset: {RAW_PATH}"

TARGET_COL = "Fossil CO2 Emissions (tons)"

# Step 1: Initial Data Exploration
df = pd.read_csv(RAW_PATH)
df.columns = [c.strip() for c in df.columns]
print("Shape:", df.shape)
display(df.head())

print("Columns:")
print(list(df.columns))

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

print("\nBasic stats (numeric):")
display(df.describe(include=[np.number]))


## Step 2: Handling Missing Values

For this dataset, we typically keep the rows and rely on the imputer later. Here we at least ensure `Year` exists.

In [ ]:
df_work = df.copy()

# Year is required for time-based splitting
df_work["Year"] = pd.to_numeric(df_work["Year"], errors="coerce")
before = len(df_work)
df_work = df_work.dropna(subset=["Year"]).copy()
print(f"Rows dropped due to missing/invalid Year: {before - len(df_work):,}")

print("Remaining missing values:")
display(df_work.isna().sum().sort_values(ascending=False))


## Step 3: Dealing with Duplicates

We remove duplicates by `Year` (time series should be unique per year).

In [ ]:
dup_rows = df_work.duplicated().sum()
print("Duplicate rows (full-row):", dup_rows)

df_work = df_work.drop_duplicates(subset=["Year"], keep="first").copy()
print("Shape after dropping Year duplicates:", df_work.shape)


## Step 4: Handling Outliers

We cap numeric outliers using IQR for all numeric columns except `Year`.

In [ ]:
def cap_outliers_iqr(s: pd.Series, k: float = 1.5) -> pd.Series:
    if s.dropna().empty:
        return s
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        return s
    low = q1 - k * iqr
    high = q3 + k * iqr
    return s.clip(low, high)

for c in df_work.columns:
    if c == "Year":
        continue
    df_work[c] = pd.to_numeric(df_work[c], errors="coerce")

numeric_cols = [c for c in df_work.columns if c != "Year" and pd.api.types.is_numeric_dtype(df_work[c])]
for c in numeric_cols:
    df_work[c] = cap_outliers_iqr(df_work[c])

print("Outlier capping done for:", numeric_cols)


## Step 5: Data Manipulation (Lag Features)

Create lag-1 features for key variables so the model can use temporal patterns.
This is a simple time-series feature engineering approach.

In [ ]:
assert TARGET_COL in df_work.columns, f"Missing target: {TARGET_COL}"

df_work = df_work.sort_values("Year").copy()

LAG_COLS = [
    TARGET_COL,
    "CO2 emissions per capita",
    "World Population",
    "GDP Real (USD)",
    "GDP growth (%)",
]
LAG_COLS = [c for c in LAG_COLS if c in df_work.columns]

for c in LAG_COLS:
    df_work[f"{c} (lag1)"] = df_work[c].shift(1)

lag_feature_cols = [f"{c} (lag1)" for c in LAG_COLS]
df_model = df_work.dropna(subset=lag_feature_cols).copy()
print("Shape after creating lag features:", df_model.shape)


## Step 6: Partitioning Dataset (Time-aware split)

Use the last 20% of years as test.

In [ ]:
test_fraction = 0.2
n_test = max(1, int(len(df_model) * test_fraction))

df_model = df_model.sort_values("Year")
df_train = df_model.iloc[:-n_test].copy()
df_test = df_model.iloc[-n_test:].copy()

print("Train size:", len(df_train), "Test size:", len(df_test))

feature_cols = [c for c in df_model.columns if c not in ["Year", TARGET_COL]]
X_train = df_train[feature_cols]
X_test = df_test[feature_cols]
y_train = df_train[TARGET_COL]
y_test = df_test[TARGET_COL]


## Step 7: Feature Scaling + Preprocessing


In [ ]:
numeric_features = X_train.columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

preprocessor = ColumnTransformer(
    transformers=[("num", numeric_transformer, numeric_features)],
    remainder="drop",
)

X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print("X_train_prep shape:", X_train_prep.shape)
print("X_test_prep shape:", X_test_prep.shape)

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_out_path = PROCESSED_DIR / "6-world_gdp_population_co2_emissions_cleaned.csv"
df_model.to_csv(df_out_path, index=False)
print("Saved modeling dataframe to:", df_out_path)

prep_out_path = PROCESSED_DIR / "6-world_preprocessor.joblib"
joblib.dump(preprocessor, prep_out_path)
print("Saved preprocessor to:", prep_out_path)


## Step 8: Visualizing Data


In [ ]:
plot_df = df_model.copy()

# Target over time
plt.figure(figsize=(10, 4.5))
plt.plot(plot_df["Year"], plot_df[TARGET_COL], marker="o")
plt.title(TARGET_COL)
plt.xlabel("Year")
plt.ylabel(TARGET_COL)
plt.tight_layout()
plt.show()

# Correlation heatmap
corr_cols = plot_df.drop(columns=["Year"], errors="ignore").select_dtypes(include=[np.number]).columns
plt.figure(figsize=(10, 7))
sns.heatmap(plot_df[corr_cols].corr(), cmap="coolwarm", center=0)
plt.title("Feature Correlations")
plt.tight_layout()
plt.show()

# Scatter plots vs target
scatter_cols = ["GDP Real (USD)", "World Population", "CO2 emissions per capita", "Population Density (P/Km²)"]
for c in scatter_cols:
    if c in plot_df.columns:
        plt.figure(figsize=(7, 4))
        sns.scatterplot(data=plot_df, x=c, y=TARGET_COL)
        plt.title(f"{c} vs {TARGET_COL}")
        plt.tight_layout()
        plt.show()
